# GloVe — Global Vectors from co-occurrence statistics

> Tutorial pair for [`glove.py`](glove.py). Compare with
> [`word2vec.ipynb`](word2vec.ipynb).

## 1. Intuition
word2vec walks a window across text and learns from one local context at a time.
GloVe says: why not use **all** the statistics at once? Count how often every
pair of words co-occurs in the whole corpus — that giant co-occurrence matrix
already contains the meaning. GloVe then finds word vectors whose **dot product
equals the log of how often the words co-occur**. The magic comes from *ratios*:
the ratio of co-occurrence probabilities is what separates "ice" from "steam".

## 2. Concept (the slide)
- Build the co-occurrence matrix $X$, where $X_{ij}$ = (weighted) number of times
  word $j$ appears in the context of word $i$. Optionally weight a context word
  $d$ positions away by $1/d$ (closer words matter more).
- Learn word vectors $w_i$, context vectors $\tilde w_j$, and biases so that
  $$w_i^\top \tilde w_j + b_i + \tilde b_j \approx \log X_{ij}.$$
- Fit by **weighted** least squares, where the weight $f(X_{ij})$ caps the
  influence of extremely frequent pairs (like "the the").
- Optimize with **AdaGrad**; the final embedding is $w_i + \tilde w_i$.

## 3. Math derivation — why *log* co-occurrence?

Let $P_{ij}=P(j\mid i)=X_{ij}/X_i$ be the probability that word $j$ appears in the
context of $i$. Pennington et al. observe that **meaning lives in ratios**: for a
probe word $k$, the ratio $P_{ik}/P_{jk}$ is large when $k$ relates to $i$ not
$j$, small in the reverse case, and $\approx 1$ when $k$ relates to both or
neither. We want a function $F$ of the vectors to reproduce that ratio:
$$F\big(w_i,w_j,\tilde w_k\big)=\frac{P_{ik}}{P_{jk}}.$$

Vector spaces are linear, so make $F$ depend on the **difference** $w_i-w_j$, and
since the right side is a scalar, on its dot product with $\tilde w_k$:
$$F\big((w_i-w_j)^\top \tilde w_k\big)=\frac{P_{ik}}{P_{jk}}.$$

We want $F$ to turn subtraction in the argument into division on the right — that
is the homomorphism $F(a-b)=F(a)/F(b)$, whose solution is $F=\exp$. Then
$$\exp(w_i^\top \tilde w_k)=P_{ik}=\frac{X_{ik}}{X_i}
\;\Longrightarrow\; w_i^\top \tilde w_k=\log X_{ik}-\log X_i.$$
The $\log X_i$ term depends only on $i$, so absorb it (and a context-side
constant) into **bias** terms $b_i,\tilde b_k$:
$$\boxed{\,w_i^\top \tilde w_k + b_i + \tilde b_k = \log X_{ik}.\,}$$

**The weighted least-squares objective.** Turn that target into a regression,
weighting each residual by $f(X_{ij})$ so rare noisy pairs and ultra-frequent
pairs don't dominate:
$$J=\sum_{i,j} f(X_{ij})\,\big(w_i^\top \tilde w_j + b_i + \tilde b_j-\log X_{ij}\big)^2,$$
$$f(x)=\begin{cases}(x/x_{\max})^{\alpha} & x<x_{\max}\\ 1 & x\ge x_{\max}\end{cases}
\quad(\alpha=3/4,\;x_{\max}=100).$$
$f(0)=0$ skips the (huge number of) zero entries, and $f$ rises then *plateaus*
so "the" can't swamp the loss.

**Gradient.** For one entry, with residual
$r_{ij}=w_i^\top\tilde w_j+b_i+\tilde b_j-\log X_{ij}$:
$$\frac{\partial J}{\partial w_i}=f(X_{ij})\,r_{ij}\,\tilde w_j,\qquad
\frac{\partial J}{\partial \tilde w_j}=f(X_{ij})\,r_{ij}\,w_i,\qquad
\frac{\partial J}{\partial b_i}=\frac{\partial J}{\partial \tilde b_j}=f(X_{ij})\,r_{ij}.$$
GloVe trains these with **AdaGrad** (per-parameter adaptive step
$\eta/\sqrt{\sum g^2}$), which suits the wildly varying word frequencies.

## 4. NumPy implementation — co-occurrence build + AdaGrad SGD

In [ ]:
# ===== actual implementation from glove.py =====
from __future__ import annotations

from collections import Counter, defaultdict

import numpy as np

SEED = 0

def _tokenize(text: str) -> list[list[str]]:
    import re
    sents = re.split(r"[.!?]+", text.lower())
    return [s.split() for s in sents if s.split()]

class GloVeNumPy:
    r"""
    **Step 1 — co-occurrence matrix.** Slide a window over the corpus and count,
    for every (center i, context j) pair within the window, X_ij. With harmonic
    weighting a context word d positions away contributes 1/d (closer = stronger).

    **Step 2 — the objective.** GloVe fits word vectors w_i, context vectors
    \tilde w_j and biases b_i, \tilde b_j so that
        w_i · \tilde w_j + b_i + \tilde b_j  ≈  log X_ij,
    minimizing the *weighted* least-squares loss
        J = Σ_{i,j} f(X_ij) (w_i·\tilde w_j + b_i + \tilde b_j - log X_ij)^2,
    where the weighting function tames very frequent pairs:
        f(x) = (x / x_max)^alpha  if x < x_max  else  1.

    **Why log co-occurrence?** Ratios P_ik/P_jk distinguish meaning. A model whose
    parameters are linear in vectors and reproduce those ratios forces the dot
    product to equal log P_ik (+ constants) — hence the log target above.
    """

    def __init__(self, dim=50, window=5, x_max=100, alpha=0.75,
                 lr=0.05, harmonic=True, seed=SEED):
        self.dim, self.window = dim, window
        self.x_max, self.alpha = x_max, alpha
        self.lr, self.harmonic, self.seed = lr, harmonic, seed

    def build_vocab(self, sentences):
        counts = Counter(w for s in sentences for w in s)
        self.itos = list(counts)
        self.stoi = {w: i for i, w in enumerate(self.itos)}
        return self

    def build_cooccurrence(self, sentences):
        """Symmetric co-occurrence counts X_ij (optionally 1/distance weighted)."""
        X = defaultdict(float)
        for s in sentences:
            ids = [self.stoi[w] for w in s if w in self.stoi]
            for i, ci in enumerate(ids):
                lo = max(0, i - self.window)
                for j in range(lo, i):              # left context only -> add both ways
                    d = i - j
                    inc = (1.0 / d) if self.harmonic else 1.0
                    X[(ci, ids[j])] += inc
                    X[(ids[j], ci)] += inc          # keep it symmetric
        # store as parallel arrays for fast SGD
        self.coo_i = np.array([k[0] for k in X], dtype=np.int64)
        self.coo_j = np.array([k[1] for k in X], dtype=np.int64)
        self.coo_x = np.array(list(X.values()), dtype=np.float64)
        return self

    def _f(self, x):
        # weighting function f(X_ij): cap influence of frequent pairs
        return np.where(x < self.x_max, (x / self.x_max) ** self.alpha, 1.0)

    def fit(self, sentences, epochs=50):
        self.build_vocab(sentences).build_cooccurrence(sentences)
        rng = np.random.default_rng(self.seed)
        V = len(self.itos)
        s = 0.5 / self.dim
        self.W = (rng.random((V, self.dim)) - 0.5) * s   # word vectors
        self.Wt = (rng.random((V, self.dim)) - 0.5) * s  # context vectors
        self.b = np.zeros(V)
        self.bt = np.zeros(V)
        # AdaGrad accumulators (sum of squared grads) — start at 1 to bound step
        gW = np.ones_like(self.W); gWt = np.ones_like(self.Wt)
        gb = np.ones_like(self.b); gbt = np.ones_like(self.bt)

        logx = np.log(self.coo_x)
        fx = self._f(self.coo_x)
        n = len(self.coo_x)
        self.history = []
        for _ in range(epochs):
            perm = rng.permutation(n)
            total = 0.0
            for idx in perm:
                i, j = self.coo_i[idx], self.coo_j[idx]
                # prediction error: w_i·w~_j + b_i + b~_j - log X_ij
                diff = self.W[i] @ self.Wt[j] + self.b[i] + self.bt[j] - logx[idx]
                w = fx[idx]
                total += 0.5 * w * diff * diff
                g = w * diff                          # shared scalar gradient
                # gradients
                gradW = g * self.Wt[j]
                gradWt = g * self.W[i]
                # AdaGrad updates: lr / sqrt(accumulated sq grad)
                gW[i] += gradW ** 2; gWt[j] += gradWt ** 2
                gb[i] += g * g; gbt[j] += g * g
                self.W[i] -= self.lr * gradW / np.sqrt(gW[i])
                self.Wt[j] -= self.lr * gradWt / np.sqrt(gWt[j])
                self.b[i] -= self.lr * g / np.sqrt(gb[i])
                self.bt[j] -= self.lr * g / np.sqrt(gbt[j])
            self.history.append(total / n)
        # GloVe uses the SUM/average of the two vector sets as the final embedding
        self.embeddings = self.W + self.Wt
        return self

    def vec(self, w):
        return self.embeddings[self.stoi[w]]

    def most_similar(self, w, k=5):
        q = self.vec(w); q = q / (np.linalg.norm(q) + 1e-9)
        M = self.embeddings / (np.linalg.norm(self.embeddings, axis=1, keepdims=True) + 1e-9)
        sim = M @ q
        order = np.argsort(-sim)
        return [(self.itos[i], float(sim[i])) for i in order if self.itos[i] != w][:k]

## 5. PyTorch implementation — same objective via autograd + `Adagrad`

In [ ]:
# ===== actual implementation from glove.py =====
import torch

import torch.nn as nn

def get_device():
    """CUDA > MPS > CPU."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

class GloVeTorch(nn.Module):
    def __init__(self, vocab, dim=50, x_max=100, alpha=0.75):
        super().__init__()
        self.W = nn.Embedding(vocab, dim)
        self.Wt = nn.Embedding(vocab, dim)
        self.b = nn.Embedding(vocab, 1)
        self.bt = nn.Embedding(vocab, 1)
        for p in (self.W, self.Wt):
            nn.init.uniform_(p.weight, -0.5 / dim, 0.5 / dim)
        nn.init.zeros_(self.b.weight); nn.init.zeros_(self.bt.weight)
        self.x_max, self.alpha = x_max, alpha

    def forward(self, i, j, x):
        pred = (self.W(i) * self.Wt(j)).sum(-1) + self.b(i).squeeze(-1) + self.bt(j).squeeze(-1)
        f = torch.where(x < self.x_max, (x / self.x_max) ** self.alpha, torch.ones_like(x))
        return (f * (pred - torch.log(x)) ** 2).mean()

def train_glove_torch(glove_np, dim=50, epochs=80, lr=0.05):
    """Reuse the NumPy object's vocab + co-occurrence arrays to train torch."""
    dev = get_device()
    model = GloVeTorch(len(glove_np.itos), dim, glove_np.x_max, glove_np.alpha).to(dev)
    opt = torch.optim.Adagrad(model.parameters(), lr=lr)
    i = torch.as_tensor(glove_np.coo_i, device=dev)
    j = torch.as_tensor(glove_np.coo_j, device=dev)
    x = torch.as_tensor(glove_np.coo_x, dtype=torch.float32, device=dev)
    for _ in range(epochs):
        opt.zero_grad(); loss = model(i, j, x); loss.backward(); opt.step()
    emb = (model.W.weight + model.Wt.weight).detach().cpu().numpy()
    return model, emb

def toy_corpus() -> str:
    # Content-only sentences: animal words only co-occur with animal words,
    # fruit words only with fruit words. No shared filler -> clean topic split.
    animals = ("dog cat lion tiger wolf bark hunt run chase prey fur paws. "
               "dog cat hunt prey. lion tiger run chase. wolf dog bark hunt. "
               "cat tiger fur paws. lion wolf chase prey run. ")
    fruits = ("apple banana orange grape mango sweet juicy ripe peel seed grow. "
              "apple banana sweet juicy. orange grape ripe grow. mango apple peel seed. "
              "banana mango sweet grow. grape orange juicy ripe peel. ")
    s = ""
    for _ in range(60):
        s += animals + fruits
    return s

def demo():
    np.random.seed(SEED); torch.manual_seed(SEED)
    sents = _tokenize(toy_corpus())

    g = GloVeNumPy(dim=20, window=3, x_max=30, lr=0.05).fit(sents, epochs=150)
    print(f"Co-occurrence entries: {len(g.coo_x)}  |  vocab: {len(g.itos)}")
    print(f"Final weighted-LS loss: {g.history[-1]:.4f}")
    print("NumPy GloVe nearest neighbours:")
    for q in ("dog", "apple"):
        print(f"  {q:6s} -> {[t for t, _ in g.most_similar(q, 3)]}")

    # sanity: the reconstruction w_i·w~_j + b ≈ log X_ij for a frequent pair
    i, j = g.stoi["dog"], g.stoi["cat"]
    pred = g.W[i] @ g.Wt[j] + g.b[i] + g.bt[j]
    import math
    actual = None
    for a, b, x in zip(g.coo_i, g.coo_j, g.coo_x):
        if a == i and b == j:
            actual = math.log(x); break
    if actual is not None:
        print(f"\nReconstruct log X['dog','cat']: pred={pred:.3f} vs log X={actual:.3f}")

    _, emb = train_glove_torch(g, dim=20, epochs=120)
    print(f"\nTorch GloVe trained (AdaGrad); embedding shape = {emb.shape}")
    fruit_words = set("apple banana orange grape mango sweet juicy ripe peel seed grow".split())
    nn3 = {t for t, _ in g.most_similar("apple", 3)}
    print(f"'apple' neighbours all in fruit topic: {nn3 <= fruit_words}")

## 6. Train / run — fit on a toy corpus and recover log X_ij

In [ ]:
demo()

## 7. Visualization — embeddings split by topic (PCA to 2-D)

In [ ]:
import matplotlib
matplotlib.use("Agg")
import numpy as np, matplotlib.pyplot as plt
import glove as M

sents = M._tokenize(M.toy_corpus())
g = M.GloVeNumPy(dim=20, window=3, x_max=30, lr=0.05).fit(sents, epochs=150)
words = list(g.itos)
V = np.stack([g.vec(t) for t in words]); V = V - V.mean(0)
_, _, Vt = np.linalg.svd(V, full_matrices=False)   # PCA via SVD
P = V @ Vt[:2].T

animals = set("dog cat lion tiger wolf bark hunt run chase prey fur paws".split())
plt.figure(figsize=(6.5, 5))
for (x, y), t in zip(P, words):
    color = "tab:red" if t in animals else "tab:green"
    plt.scatter(x, y, color=color); plt.annotate(t, (x, y), fontsize=8)
plt.title("GloVe embeddings (red = animal topic, green = fruit topic)")
plt.tight_layout(); plt.show()

## 8. Takeaways & pitfalls
- GloVe = **matrix factorization of log co-occurrence** with a clever weighting;
  word2vec is implicitly factorizing a (shifted-PMI) matrix too, so the two are
  close cousins reaching similar geometry from opposite directions
  (global counts vs local windows).
- The **weighting $f(X)$** is essential: without it, function words dominate; with
  $f(0)=0$ you also skip the overwhelmingly common zero entries.
- Use the **sum** $w_i+\tilde w_i$ as the final vector — it averages out noise.
- Static embeddings still give one vector per word — context-dependent meaning
  needs [Transformers](../../05.transformers/architectures/transformer.ipynb).
- On a *tiny* toy corpus the co-occurrence matrix is sparse and noisy; real GloVe
  shines on billions of tokens.